# TabICL - fast-model EDA on Kaggle GPU

Reads an untuned **[TabICL](https://github.com/soda-inria/tabicl)** (Qu et al., 2025 -
in-context learning that scales to large tabular data) score as a second opinion on the
*rough achievable score* for churn, next to the untuned LightGBM baseline (~0.91 ROC-AUC)
and our tuned best (~0.92).

**Why Kaggle + subsample:** TabICL is an ICL transformer (GPU). It is strong from ~300 to
~100k rows (and can reach ~500k via CPU/disk offloading, slower). We feed the harness a
stratified 50k subsample on a T4. SHAP is skipped - TabICL is not a tree.

**Settings (right sidebar):** Accelerator -> GPU T4 x2 (not P100); Internet -> On;
Add Input -> playground-series-s6e3.

> WARNING: This scaffold is **not verified against the live environment**. Confirm
> `TabICLClassifier`'s class name / kwargs against the installed `tabicl` version, and
> **smoke-test one fold** (`n_splits=2`) first.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell. (A plain
# `if REPO_ROOT not in sys.path` guard can leave a shadowing `src` ahead of ours.)
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


Updating files: 100% (132/132), done.


In [3]:
!pip install -q tabicl

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 114.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu1

In [4]:
# cuda.is_available() can return True on an incompatible GPU - run a real op.
import torch
print("device:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
try:
    _ = (torch.randn(16, device="cuda") @ torch.randn(16, 16, device="cuda")).sum().item()
    print("GPU compute OK")
except Exception as e:
    print("GPU compute FAILED:", e)            # if this fails, switch to T4 and restart

from tabicl import TabICLClassifier
print("TabICLClassifier imported OK")

device: Tesla T4 | count: 2
GPU compute OK
TabICLClassifier imported OK


In [5]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild them
# from the attached competition CSVs.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(force=True)   # writes data/processed/*.parquet

Preprocessed and saved: train_df (594194, 42), test_df (254655, 41)


In [6]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score

from src.tracking import RUNS_DIR
from src.cv import run_cv_experiment, save_experiment

### Design matrix and TabICL subsample

`prepare_data` returns the one-hot-encoded frames. TabICL is an in-context model, so its
"training" is the context it attends to at predict time. It is strong up to ~100k rows
(more with offloading); we hand the harness a stratified `SUBSAMPLE_N` sample, so the OOF
ROC-AUC is computed over that subsample. Each fold predicts the full 254k test set - the
slow part on a T4; reduce `SUBSAMPLE_N` or `n_splits` if memory or time is tight.

In [7]:
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

# TabICL is strong from ~300 to ~100k rows (more via CPU/disk offloading). Subsample
# within the GPU-comfortable range; raise toward 100k+ if the T4 has headroom.
SUBSAMPLE_N = 50_000
X_sub, _, y_sub, _ = train_test_split(
    X_train, y_train, train_size=SUBSAMPLE_N, stratify=y_train, random_state=42,
)
X_sub = X_sub.reset_index(drop=True)
y_sub = y_sub.reset_index(drop=True)
print(f'Subsample: {X_sub.shape}  churn rate: {y_sub.mean():.3f}  (full: {y_train.mean():.3f})')

Subsample: (50000, 40)  churn rate: 0.225  (full: 0.225)


### Run configuration - TabICL baseline

Same `run_config` shape as the other runs, fed the subsample. `metric=accuracy_score`;
the harness *always* logs **OOF ROC-AUC** separately (the project's primary metric), so
that is the achievable-score signal we read. `save_models=False` (torch-backed).

In [8]:
import numpy as np
from tabicl import TabICLClassifier


class ChunkedTabICL(TabICLClassifier):
    """Predict the test set in row-chunks so TabICL's per-call output buffer
    stays within T4 VRAM.

    The full 254k-row test set makes TabICL try to allocate one ~22 GB
    column-embedding buffer (shape ~ (n_query, seq_len, out_dim)); on a T4 that
    overflows GPU -> CPU -> disk and raises an OOM unless disk_offload_dir is
    set. Chunking predict_proba is exact -- each query row attends to the same
    fitted context independently, so chunk boundaries don't change the output --
    and keeps each call's buffer small (~1.7 GB at CHUNK=20k), entirely on GPU.
    Fit-time context (SUBSAMPLE_N) is a separate axis and is unaffected.
    """
    CHUNK = 20_000

    def predict_proba(self, X):
        if len(X) <= self.CHUNK:
            return super().predict_proba(X)
        sup = super(ChunkedTabICL, self)   # bind once: zero-arg super() breaks inside a loop
        parts = []
        for i in range(0, len(X), self.CHUNK):
            Xi = X.iloc[i:i + self.CHUNK] if hasattr(X, "iloc") else X[i:i + self.CHUNK]
            parts.append(sup.predict_proba(Xi))
        return np.vstack(parts)


tabicl_params = {
    'device':       'cuda',
    'random_state': 42,
}

DATA_VERSION = 'fe_v0'   # identity FE (no engineered features) - baseline

run_config = {
    'model_factory': lambda params: ChunkedTabICL(**params),   # chunked test prediction
    'params':        tabicl_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'tabicl-eda',
    'notes': (
        'TabICL (untuned TabICLClassifier) fast-EDA pass on Kaggle T4 GPU, stratified '
        '50k subsample. OOF ROC-AUC read as a second opinion vs untuned LGBM ~0.91. '
        'tabicl=2.1.1, torch=2.10.0+cu128. Data regenerated on-platform; GPU not bit-reproducible.'
    ),
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [9]:
# Step 1 - Run the experiment. SMOKE-TEST FIRST: set n_splits=2 in the run_config cell,
# confirm the GPU path works and check per-fold time, before a full Save & Run All.
result = run_cv_experiment(run_config, X_sub, y_sub, X_test, encoded_features)

Run ID: 20260603-222236-617994
Tag:    tabicl-eda

INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt:   0%|          | 0.00/110M [00:00<?, ?B/s]

Fold 0: accuracy=0.8590  roc_auc=0.9144  (fit 3.8s)
Fold 1: accuracy=0.8590  roc_auc=0.9151  (fit 1.8s)
Fold 2: accuracy=0.8579  roc_auc=0.9102  (fit 1.8s)
Fold 3: accuracy=0.8568  roc_auc=0.9109  (fit 1.8s)
Fold 4: accuracy=0.8606  roc_auc=0.9117  (fit 1.7s)

OOF accuracy: 0.8587
OOF ROC-AUC:  0.9124
Folds:        0.8587 ± 0.0013

Run complete. Call save_experiment(result) to log this run permanently.


In [10]:
# Step 2 - Save the run (optional). Review the OOF ROC-AUC printed above first.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260603-222236-617994


### Build a submission (optional)

`test_proba_mean` is the fold-bagged churn probability for the full test set. The
competition metric is ROC-AUC, so submit the probability directly.

In [11]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.039709
1  594195  0.000461
2  594196  0.065525
3  594197  0.002185
4  594198  0.460219
wrote /kaggle/working/submission.csv (254655, 2)


### Artifact hand-off

To fold this run back into the local repo — extract the run dir, append the new
`runs.csv` row, optionally submit and backfill `lb_public` / `lb_private`, and commit
this notebook under `kaggle/` — follow **§7-8 of `docs/kaggle_gpu_workflow.md`**.
Start by zipping the run directory for download (Output tab):

    import shutil
    from src.tracking import RUNS_DIR
    shutil.make_archive(f"/kaggle/working/{run_id}", "zip", RUNS_DIR / run_id)

In [ ]:
    import shutil
    from src.tracking import RUNS_DIR
    shutil.make_archive(f"/kaggle/working/{run_id}", "zip", RUNS_DIR / run_id)